### RFM Segmentation Table

In [0]:
%sql
CREATE OR REPLACE TABLE dunnhumby_project.gold.rfm_segments AS
WITH cust AS (
    SELECT 
        household_key, MAX(day) AS last_day,
        COUNT(DISTINCT basket_id) AS frequency, 
        ROUND(SUM(sales_value),2) AS monetary
    FROM dunnhumby_project.silver.transactions_enriched 
    GROUP BY household_key
),
scored AS (
    SELECT *, 
    NTILE(5) OVER (ORDER BY last_day DESC) AS r_bin,
    NTILE(5) OVER (ORDER BY frequency) AS f_bin,
    NTILE(5) OVER (ORDER BY monetary) AS m_bin FROM cust
)
SELECT 
    household_key,
    CASE 
        WHEN (r_bin+m_bin)/2 >= 4 THEN 'Champions'
        WHEN (r_bin+m_bin)/2 >= 3 THEN 'Loyal'
        WHEN r_bin >= 4 AND m_bin <= 2 THEN 'New/Promising'
        WHEN r_bin <= 2 AND m_bin >= 4 THEN 'At Risk (high value)'
        ELSE 'Occasional' 
    END AS segment,
    last_day,
    frequency,
    monetary
FROM scored;

num_affected_rows,num_inserted_rows


### Customer Value Table

In [0]:
%sql

CREATE OR REPLACE TABLE dunnhumby_project.gold.customer_quintiles AS
WITH cust AS (
    SELECT household_key, 
        SUM(sales_value) AS spend
    FROM dunnhumby_project.silver.transactions_enriched 
    GROUP BY household_key
),
ranked AS (
    SELECT 
        *, 
        NTILE(5) OVER (ORDER BY spend DESC) AS quintile 
    FROM cust
)
SELECT 
    quintile, 
    COUNT(*) AS customers, 
    ROUND(SUM(spend),2) AS revenue,
    ROUND(SUM(spend)*100/SUM(SUM(spend)) OVER (),2) AS pct_revenue
FROM ranked 
GROUP BY quintile ORDER BY quintile;

num_affected_rows,num_inserted_rows


## Department Performance Table

In [0]:
%sql
CREATE OR REPLACE TABLE dunnhumby_project.gold.department_sales AS
SELECT 
    department,
    ROUND(SUM(sales_value),2) as total_sales,
    ROUND(SUM(sales_value) * 100 / SUM(SUM(sales_value)) OVER (), 2)  as percentage
FROM dunnhumby_project.silver.transactions_enriched
WHERE department NOT IN ('KIOSK-GAS','MISC SALES TRAN','COUPON/MISC')
GROUP BY department
ORDER BY total_sales DESC;

num_affected_rows,num_inserted_rows
